# Graph Traversal

```{contents}
:local:
:depth: 2
```

A service network can answer reachability, minimum handoffs, and disconnected-group questions. State direction and edge costs before choosing a traversal. The complete implementations snapshot adjacency, use ordinal labels, declare isolates explicitly, reject unknown endpoints, normalize duplicate neighbors, and preserve neighbor order. Self-loops and cycles are allowed.

## Breadth-First Discovery and Layers

```{index} Breadth-First Discovery and Layers
```

BFS uses a FIFO queue. Mark a vertex when enqueueing, not when removing it: each vertex then enters the queue once even when several predecessors discover it. Nullable distance doubles as the discovery marker; zero is the valid source distance, and null is unreachable.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;
using System.Numerics;

var adjacency=new Dictionary<string,string[]>{["HQ"]=new[]{"Warehouse","Helpdesk"},["Warehouse"]=new[]{"Depot"},["Helpdesk"]=new[]{"Client"},["Depot"]=new[]{"Client"},["Client"]=new[]{"Warehouse"},["Remote"]=Array.Empty<string>()};
var bfs=new GraphWalk(adjacency).Bfs("HQ",Console.WriteLine);
Console.WriteLine("order="+string.Join(",",bfs.Order));
Console.WriteLine($"Client layer={bfs.Distance["Client"]}, Remote unreachable={bfs.Distance["Remote"] is null}, arcs examined={bfs.EdgesExamined}");
public sealed class GraphWalk
{
    private readonly Dictionary<string,string[]> graph;
    private readonly string[] vertices;
    public GraphWalk(Dictionary<string,string[]> adjacency)
    {
        ArgumentNullException.ThrowIfNull(adjacency);
        graph=new Dictionary<string,string[]>(StringComparer.Ordinal);
        foreach(var pair in adjacency)
        {
            if(pair.Value is null)throw new ArgumentException("Null adjacency list.");
            graph.Add(pair.Key,pair.Value.Distinct(StringComparer.Ordinal).ToArray());
        }
        foreach(var neighbors in graph.Values)
            if(neighbors.Any(v=>v is null||!graph.ContainsKey(v)))throw new ArgumentException("Unknown endpoint.");
        vertices=graph.Keys.OrderBy(v=>v,StringComparer.Ordinal).ToArray();
    }
    public sealed class Layers
    {
        public string Source { get; }
        public string[] Order { get; }
        public IReadOnlyDictionary<string,int?> Distance { get; }
        public IReadOnlyDictionary<string,string?> Previous { get; }
        public long EdgesExamined { get; }
        internal Layers(string source,string[] order,Dictionary<string,int?> distance,Dictionary<string,string?> previous,long edges)
        {
            Source=source;Order=order;EdgesExamined=edges;
            Distance=new System.Collections.ObjectModel.ReadOnlyDictionary<string,int?>(distance);
            Previous=new System.Collections.ObjectModel.ReadOnlyDictionary<string,string?>(previous);
        }
        public string[] PathTo(string target)
        {
            if(!Distance.ContainsKey(target))throw new KeyNotFoundException(target);
            if(Distance[target] is null)return Array.Empty<string>();
            var path=new Stack<string>();string? at=target;
            while(at is not null)
            {
                if(path.Count>=Distance.Count)throw new InvalidOperationException("Predecessor cycle.");
                path.Push(at);at=Previous[at];
            }
            if(path.Peek()!=Source)throw new InvalidOperationException("Invalid source path.");
            return path.ToArray();
        }
    }
    public Layers Bfs(string source,Action<string>? trace=null)
    {
        if(!graph.ContainsKey(source))throw new KeyNotFoundException(source);
        var distance=vertices.ToDictionary(v=>v,_=>(int?)null,StringComparer.Ordinal);
        var previous=vertices.ToDictionary(v=>v,_=>(string?)null,StringComparer.Ordinal);
        var queue=new Queue<string>();var order=new List<string>();long edges=0;
        distance[source]=0;queue.Enqueue(source);
        while(queue.Count>0)
        {
            string at=queue.Dequeue();order.Add(at);trace?.Invoke($"visit {at}: layer={distance[at]}");
            foreach(string next in graph[at])
            {
                edges++;
                if(distance[next] is not null)continue;
                distance[next]=distance[at]!.Value+1;previous[next]=at;queue.Enqueue(next);
            }
        }
        return new Layers(source,order.ToArray(),distance,previous,edges);
    }
    public record Depth(string[] Preorder,string[] FinishOrder,long EdgesExamined);
    public Depth Dfs(string source,Action<string>? trace=null)
    {
        if(!graph.ContainsKey(source))throw new KeyNotFoundException(source);
        var color=vertices.ToDictionary(v=>v,_=>0,StringComparer.Ordinal);
        var stack=new Stack<(string Vertex,int Next)>();var pre=new List<string>();var finish=new List<string>();long edges=0;
        Enter(source);
        while(stack.Count>0)
        {
            var frame=stack.Pop();
            if(frame.Next==graph[frame.Vertex].Length)
            {color[frame.Vertex]=2;finish.Add(frame.Vertex);trace?.Invoke($"exit {frame.Vertex}");continue;}
            stack.Push((frame.Vertex,frame.Next+1));string next=graph[frame.Vertex][frame.Next];edges++;
            if(color[next]==0)Enter(next);
            else if(color[next]==1)trace?.Invoke($"active edge {frame.Vertex}->{next}");
        }
        return new Depth(pre.ToArray(),finish.ToArray(),edges);
        void Enter(string vertex){color[vertex]=1;pre.Add(vertex);stack.Push((vertex,0));trace?.Invoke($"enter {vertex}");}
    }
    public string[][] UndirectedComponents()
    {
        var neighbors=graph.ToDictionary(p=>p.Key,p=>p.Value.ToHashSet(StringComparer.Ordinal),StringComparer.Ordinal);
        foreach(var pair in graph)
            foreach(string next in pair.Value)
                if(!neighbors[next].Contains(pair.Key))throw new ArgumentException("Components require symmetric adjacency.");
        var seen=new HashSet<string>(StringComparer.Ordinal);var groups=new List<string[]>();
        foreach(string root in vertices)
        {
            if(!seen.Add(root))continue;
            var queue=new Queue<string>();var group=new List<string>();queue.Enqueue(root);
            while(queue.Count>0)
            {
                string at=queue.Dequeue();group.Add(at);
                foreach(string next in graph[at])if(seen.Add(next))queue.Enqueue(next);
            }
            groups.Add(group.ToArray());
        }
        return groups.ToArray();
    }
}


Queue entries have nondecreasing distance; undiscovered neighbors of a layer-d vertex receive d+1. If a shorter path to a newly discovered vertex existed, its predecessor in an earlier layer would already have discovered it. Induction on layers proves first discovery gives minimum edge count. Ties depend on supplied neighbor order; shortest distance is unique as a number even when several shortest paths exist.

## Reconstruct Only Reachable Paths

```{index} Reconstruct Only Reachable Paths
```

Record the predecessor on first discovery. Never fabricate a singleton target path for an unreachable vertex, and distinguish an unknown label from a known unreachable one.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;
using System.Numerics;

var adjacency=new Dictionary<string,string[]>{["HQ"]=new[]{"Warehouse","Helpdesk"},["Warehouse"]=new[]{"Depot"},["Helpdesk"]=new[]{"Client"},["Depot"]=new[]{"Client"},["Client"]=new[]{"Warehouse"},["Remote"]=Array.Empty<string>()};
var routes=new GraphWalk(adjacency).Bfs("HQ");
foreach(string target in new[]{"HQ","Client","Remote"})Console.WriteLine($"{target}: [{string.Join("->",routes.PathTo(target))}]");
try{routes.PathTo("Missing");}catch(KeyNotFoundException){Console.WriteLine("unknown target rejected");}
public sealed class GraphWalk
{
    private readonly Dictionary<string,string[]> graph;
    private readonly string[] vertices;
    public GraphWalk(Dictionary<string,string[]> adjacency)
    {
        ArgumentNullException.ThrowIfNull(adjacency);
        graph=new Dictionary<string,string[]>(StringComparer.Ordinal);
        foreach(var pair in adjacency)
        {
            if(pair.Value is null)throw new ArgumentException("Null adjacency list.");
            graph.Add(pair.Key,pair.Value.Distinct(StringComparer.Ordinal).ToArray());
        }
        foreach(var neighbors in graph.Values)
            if(neighbors.Any(v=>v is null||!graph.ContainsKey(v)))throw new ArgumentException("Unknown endpoint.");
        vertices=graph.Keys.OrderBy(v=>v,StringComparer.Ordinal).ToArray();
    }
    public sealed class Layers
    {
        public string Source { get; }
        public string[] Order { get; }
        public IReadOnlyDictionary<string,int?> Distance { get; }
        public IReadOnlyDictionary<string,string?> Previous { get; }
        public long EdgesExamined { get; }
        internal Layers(string source,string[] order,Dictionary<string,int?> distance,Dictionary<string,string?> previous,long edges)
        {
            Source=source;Order=order;EdgesExamined=edges;
            Distance=new System.Collections.ObjectModel.ReadOnlyDictionary<string,int?>(distance);
            Previous=new System.Collections.ObjectModel.ReadOnlyDictionary<string,string?>(previous);
        }
        public string[] PathTo(string target)
        {
            if(!Distance.ContainsKey(target))throw new KeyNotFoundException(target);
            if(Distance[target] is null)return Array.Empty<string>();
            var path=new Stack<string>();string? at=target;
            while(at is not null)
            {
                if(path.Count>=Distance.Count)throw new InvalidOperationException("Predecessor cycle.");
                path.Push(at);at=Previous[at];
            }
            if(path.Peek()!=Source)throw new InvalidOperationException("Invalid source path.");
            return path.ToArray();
        }
    }
    public Layers Bfs(string source,Action<string>? trace=null)
    {
        if(!graph.ContainsKey(source))throw new KeyNotFoundException(source);
        var distance=vertices.ToDictionary(v=>v,_=>(int?)null,StringComparer.Ordinal);
        var previous=vertices.ToDictionary(v=>v,_=>(string?)null,StringComparer.Ordinal);
        var queue=new Queue<string>();var order=new List<string>();long edges=0;
        distance[source]=0;queue.Enqueue(source);
        while(queue.Count>0)
        {
            string at=queue.Dequeue();order.Add(at);trace?.Invoke($"visit {at}: layer={distance[at]}");
            foreach(string next in graph[at])
            {
                edges++;
                if(distance[next] is not null)continue;
                distance[next]=distance[at]!.Value+1;previous[next]=at;queue.Enqueue(next);
            }
        }
        return new Layers(source,order.ToArray(),distance,previous,edges);
    }
    public record Depth(string[] Preorder,string[] FinishOrder,long EdgesExamined);
    public Depth Dfs(string source,Action<string>? trace=null)
    {
        if(!graph.ContainsKey(source))throw new KeyNotFoundException(source);
        var color=vertices.ToDictionary(v=>v,_=>0,StringComparer.Ordinal);
        var stack=new Stack<(string Vertex,int Next)>();var pre=new List<string>();var finish=new List<string>();long edges=0;
        Enter(source);
        while(stack.Count>0)
        {
            var frame=stack.Pop();
            if(frame.Next==graph[frame.Vertex].Length)
            {color[frame.Vertex]=2;finish.Add(frame.Vertex);trace?.Invoke($"exit {frame.Vertex}");continue;}
            stack.Push((frame.Vertex,frame.Next+1));string next=graph[frame.Vertex][frame.Next];edges++;
            if(color[next]==0)Enter(next);
            else if(color[next]==1)trace?.Invoke($"active edge {frame.Vertex}->{next}");
        }
        return new Depth(pre.ToArray(),finish.ToArray(),edges);
        void Enter(string vertex){color[vertex]=1;pre.Add(vertex);stack.Push((vertex,0));trace?.Invoke($"enter {vertex}");}
    }
    public string[][] UndirectedComponents()
    {
        var neighbors=graph.ToDictionary(p=>p.Key,p=>p.Value.ToHashSet(StringComparer.Ordinal),StringComparer.Ordinal);
        foreach(var pair in graph)
            foreach(string next in pair.Value)
                if(!neighbors[next].Contains(pair.Key))throw new ArgumentException("Components require symmetric adjacency.");
        var seen=new HashSet<string>(StringComparer.Ordinal);var groups=new List<string[]>();
        foreach(string root in vertices)
        {
            if(!seen.Add(root))continue;
            var queue=new Queue<string>();var group=new List<string>();queue.Enqueue(root);
            while(queue.Count>0)
            {
                string at=queue.Dequeue();group.Add(at);
                foreach(string next in graph[at])if(seen.Add(next))queue.Enqueue(next);
            }
            groups.Add(group.ToArray());
        }
        return groups.ToArray();
    }
}


A reachable predecessor chain decreases layer by one at each step and ends at the source. The returned path has distance+1 vertices, begins at the source, ends at the target, and follows graph arcs. Path reconstruction costs O(path length) time/output storage. Distance/predecessor maps are read-only; adjacency input changes after construction do not change the owned graph.

## Depth-First Entry and Exit

```{index} Depth-First Entry and Exit
```

DFS explores a child completely before the next sibling. Recursive calls implicitly store the current vertex and next neighbor; an explicit frame stack can store the same information without a deep process call stack. Colors distinguish undiscovered, active, and finished vertices.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;
using System.Numerics;

var adjacency=new Dictionary<string,string[]>{["HQ"]=new[]{"Warehouse","Helpdesk"},["Warehouse"]=new[]{"Depot"},["Helpdesk"]=new[]{"Client"},["Depot"]=new[]{"Client"},["Client"]=new[]{"Warehouse"},["Remote"]=Array.Empty<string>()};
var dfs=new GraphWalk(adjacency).Dfs("HQ",Console.WriteLine);
Console.WriteLine("preorder="+string.Join(",",dfs.Preorder));
Console.WriteLine("finish="+string.Join(",",dfs.FinishOrder));
public sealed class GraphWalk
{
    private readonly Dictionary<string,string[]> graph;
    private readonly string[] vertices;
    public GraphWalk(Dictionary<string,string[]> adjacency)
    {
        ArgumentNullException.ThrowIfNull(adjacency);
        graph=new Dictionary<string,string[]>(StringComparer.Ordinal);
        foreach(var pair in adjacency)
        {
            if(pair.Value is null)throw new ArgumentException("Null adjacency list.");
            graph.Add(pair.Key,pair.Value.Distinct(StringComparer.Ordinal).ToArray());
        }
        foreach(var neighbors in graph.Values)
            if(neighbors.Any(v=>v is null||!graph.ContainsKey(v)))throw new ArgumentException("Unknown endpoint.");
        vertices=graph.Keys.OrderBy(v=>v,StringComparer.Ordinal).ToArray();
    }
    public sealed class Layers
    {
        public string Source { get; }
        public string[] Order { get; }
        public IReadOnlyDictionary<string,int?> Distance { get; }
        public IReadOnlyDictionary<string,string?> Previous { get; }
        public long EdgesExamined { get; }
        internal Layers(string source,string[] order,Dictionary<string,int?> distance,Dictionary<string,string?> previous,long edges)
        {
            Source=source;Order=order;EdgesExamined=edges;
            Distance=new System.Collections.ObjectModel.ReadOnlyDictionary<string,int?>(distance);
            Previous=new System.Collections.ObjectModel.ReadOnlyDictionary<string,string?>(previous);
        }
        public string[] PathTo(string target)
        {
            if(!Distance.ContainsKey(target))throw new KeyNotFoundException(target);
            if(Distance[target] is null)return Array.Empty<string>();
            var path=new Stack<string>();string? at=target;
            while(at is not null)
            {
                if(path.Count>=Distance.Count)throw new InvalidOperationException("Predecessor cycle.");
                path.Push(at);at=Previous[at];
            }
            if(path.Peek()!=Source)throw new InvalidOperationException("Invalid source path.");
            return path.ToArray();
        }
    }
    public Layers Bfs(string source,Action<string>? trace=null)
    {
        if(!graph.ContainsKey(source))throw new KeyNotFoundException(source);
        var distance=vertices.ToDictionary(v=>v,_=>(int?)null,StringComparer.Ordinal);
        var previous=vertices.ToDictionary(v=>v,_=>(string?)null,StringComparer.Ordinal);
        var queue=new Queue<string>();var order=new List<string>();long edges=0;
        distance[source]=0;queue.Enqueue(source);
        while(queue.Count>0)
        {
            string at=queue.Dequeue();order.Add(at);trace?.Invoke($"visit {at}: layer={distance[at]}");
            foreach(string next in graph[at])
            {
                edges++;
                if(distance[next] is not null)continue;
                distance[next]=distance[at]!.Value+1;previous[next]=at;queue.Enqueue(next);
            }
        }
        return new Layers(source,order.ToArray(),distance,previous,edges);
    }
    public record Depth(string[] Preorder,string[] FinishOrder,long EdgesExamined);
    public Depth Dfs(string source,Action<string>? trace=null)
    {
        if(!graph.ContainsKey(source))throw new KeyNotFoundException(source);
        var color=vertices.ToDictionary(v=>v,_=>0,StringComparer.Ordinal);
        var stack=new Stack<(string Vertex,int Next)>();var pre=new List<string>();var finish=new List<string>();long edges=0;
        Enter(source);
        while(stack.Count>0)
        {
            var frame=stack.Pop();
            if(frame.Next==graph[frame.Vertex].Length)
            {color[frame.Vertex]=2;finish.Add(frame.Vertex);trace?.Invoke($"exit {frame.Vertex}");continue;}
            stack.Push((frame.Vertex,frame.Next+1));string next=graph[frame.Vertex][frame.Next];edges++;
            if(color[next]==0)Enter(next);
            else if(color[next]==1)trace?.Invoke($"active edge {frame.Vertex}->{next}");
        }
        return new Depth(pre.ToArray(),finish.ToArray(),edges);
        void Enter(string vertex){color[vertex]=1;pre.Add(vertex);stack.Push((vertex,0));trace?.Invoke($"enter {vertex}");}
    }
    public string[][] UndirectedComponents()
    {
        var neighbors=graph.ToDictionary(p=>p.Key,p=>p.Value.ToHashSet(StringComparer.Ordinal),StringComparer.Ordinal);
        foreach(var pair in graph)
            foreach(string next in pair.Value)
                if(!neighbors[next].Contains(pair.Key))throw new ArgumentException("Components require symmetric adjacency.");
        var seen=new HashSet<string>(StringComparer.Ordinal);var groups=new List<string[]>();
        foreach(string root in vertices)
        {
            if(!seen.Add(root))continue;
            var queue=new Queue<string>();var group=new List<string>();queue.Enqueue(root);
            while(queue.Count>0)
            {
                string at=queue.Dequeue();group.Add(at);
                foreach(string next in graph[at])if(seen.Add(next))queue.Enqueue(next);
            }
            groups.Add(group.ToArray());
        }
        return groups.ToArray();
    }
}


The frame stack holds the active path and each suspended neighbor index. Each adjacency entry is examined once, and each reached vertex enters/exits once. A one-shot stack that marks all neighbors immediately need not reproduce recursive DFS order. In this directed example Client→Warehouse points to an active ancestor. In an undirected graph the reverse parent edge also points to an active vertex, so this trace alone is not a general undirected cycle detector. DFS establishes reachability; its first route need not minimize handoffs: HQ→Warehouse→Depot→Client is longer than the BFS route through Helpdesk.

## Undirected Components Require Symmetry

```{index} Undirected Components Require Symmetry
```

Connected components partition an undirected graph. Repeated BFS from an unseen vertex visits one component; retain one global seen set and include isolated vertices. A directed reachability grouping depends on root order and must not be labeled undirected or strongly connected components.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;
using System.Numerics;

var adjacency=new Dictionary<string,string[]>{["HQ"]=new[]{"Shop"},["Shop"]=new[]{"HQ"},["Depot"]=new[]{"Lab"},["Lab"]=new[]{"Depot"},["Remote"]=Array.Empty<string>()};
var groups=new GraphWalk(adjacency).UndirectedComponents();
Console.WriteLine(string.Join(" | ",groups.Select(g=>"["+string.Join(",",g)+"]")));
Console.WriteLine($"components={groups.Length}, empty components={new GraphWalk(new Dictionary<string,string[]>()).UndirectedComponents().Length}");
try{new GraphWalk(new Dictionary<string,string[]>{["A"]=new[]{"B"},["B"]=Array.Empty<string>()}).UndirectedComponents();}catch(ArgumentException){Console.WriteLine("asymmetric components rejected");}
public sealed class GraphWalk
{
    private readonly Dictionary<string,string[]> graph;
    private readonly string[] vertices;
    public GraphWalk(Dictionary<string,string[]> adjacency)
    {
        ArgumentNullException.ThrowIfNull(adjacency);
        graph=new Dictionary<string,string[]>(StringComparer.Ordinal);
        foreach(var pair in adjacency)
        {
            if(pair.Value is null)throw new ArgumentException("Null adjacency list.");
            graph.Add(pair.Key,pair.Value.Distinct(StringComparer.Ordinal).ToArray());
        }
        foreach(var neighbors in graph.Values)
            if(neighbors.Any(v=>v is null||!graph.ContainsKey(v)))throw new ArgumentException("Unknown endpoint.");
        vertices=graph.Keys.OrderBy(v=>v,StringComparer.Ordinal).ToArray();
    }
    public sealed class Layers
    {
        public string Source { get; }
        public string[] Order { get; }
        public IReadOnlyDictionary<string,int?> Distance { get; }
        public IReadOnlyDictionary<string,string?> Previous { get; }
        public long EdgesExamined { get; }
        internal Layers(string source,string[] order,Dictionary<string,int?> distance,Dictionary<string,string?> previous,long edges)
        {
            Source=source;Order=order;EdgesExamined=edges;
            Distance=new System.Collections.ObjectModel.ReadOnlyDictionary<string,int?>(distance);
            Previous=new System.Collections.ObjectModel.ReadOnlyDictionary<string,string?>(previous);
        }
        public string[] PathTo(string target)
        {
            if(!Distance.ContainsKey(target))throw new KeyNotFoundException(target);
            if(Distance[target] is null)return Array.Empty<string>();
            var path=new Stack<string>();string? at=target;
            while(at is not null)
            {
                if(path.Count>=Distance.Count)throw new InvalidOperationException("Predecessor cycle.");
                path.Push(at);at=Previous[at];
            }
            if(path.Peek()!=Source)throw new InvalidOperationException("Invalid source path.");
            return path.ToArray();
        }
    }
    public Layers Bfs(string source,Action<string>? trace=null)
    {
        if(!graph.ContainsKey(source))throw new KeyNotFoundException(source);
        var distance=vertices.ToDictionary(v=>v,_=>(int?)null,StringComparer.Ordinal);
        var previous=vertices.ToDictionary(v=>v,_=>(string?)null,StringComparer.Ordinal);
        var queue=new Queue<string>();var order=new List<string>();long edges=0;
        distance[source]=0;queue.Enqueue(source);
        while(queue.Count>0)
        {
            string at=queue.Dequeue();order.Add(at);trace?.Invoke($"visit {at}: layer={distance[at]}");
            foreach(string next in graph[at])
            {
                edges++;
                if(distance[next] is not null)continue;
                distance[next]=distance[at]!.Value+1;previous[next]=at;queue.Enqueue(next);
            }
        }
        return new Layers(source,order.ToArray(),distance,previous,edges);
    }
    public record Depth(string[] Preorder,string[] FinishOrder,long EdgesExamined);
    public Depth Dfs(string source,Action<string>? trace=null)
    {
        if(!graph.ContainsKey(source))throw new KeyNotFoundException(source);
        var color=vertices.ToDictionary(v=>v,_=>0,StringComparer.Ordinal);
        var stack=new Stack<(string Vertex,int Next)>();var pre=new List<string>();var finish=new List<string>();long edges=0;
        Enter(source);
        while(stack.Count>0)
        {
            var frame=stack.Pop();
            if(frame.Next==graph[frame.Vertex].Length)
            {color[frame.Vertex]=2;finish.Add(frame.Vertex);trace?.Invoke($"exit {frame.Vertex}");continue;}
            stack.Push((frame.Vertex,frame.Next+1));string next=graph[frame.Vertex][frame.Next];edges++;
            if(color[next]==0)Enter(next);
            else if(color[next]==1)trace?.Invoke($"active edge {frame.Vertex}->{next}");
        }
        return new Depth(pre.ToArray(),finish.ToArray(),edges);
        void Enter(string vertex){color[vertex]=1;pre.Add(vertex);stack.Push((vertex,0));trace?.Invoke($"enter {vertex}");}
    }
    public string[][] UndirectedComponents()
    {
        var neighbors=graph.ToDictionary(p=>p.Key,p=>p.Value.ToHashSet(StringComparer.Ordinal),StringComparer.Ordinal);
        foreach(var pair in graph)
            foreach(string next in pair.Value)
                if(!neighbors[next].Contains(pair.Key))throw new ArgumentException("Components require symmetric adjacency.");
        var seen=new HashSet<string>(StringComparer.Ordinal);var groups=new List<string[]>();
        foreach(string root in vertices)
        {
            if(!seen.Add(root))continue;
            var queue=new Queue<string>();var group=new List<string>();queue.Enqueue(root);
            while(queue.Count>0)
            {
                string at=queue.Dequeue();group.Add(at);
                foreach(string next in graph[at])if(seen.Add(next))queue.Enqueue(next);
            }
            groups.Add(group.ToArray());
        }
        return groups.ToArray();
    }
}


Constructor copying/validation costs expected O(V+E) time and O(V+E) owned storage; sorting root labels adds O(V log V). BFS/DFS initialize maps for all V vertices and scan reached adjacency entries, so each complete traversal costs expected O(V+E), with O(V) auxiliary state. Components also validate symmetry using neighbor hash sets, giving expected O(V+E) work after construction. BFS’s frontier can contain O(V) vertices; frame DFS uses O(V) active frames in the worst case. With a matrix, scanning whole rows changes traversal work to O(V²).

## Exercise

```{index} Exercise
```

Check a singleton with a self-loop and duplicate neighbors, an isolated vertex, snapshot ownership, and invalid endpoints. Explain why discovery-time marking terminates.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;
using System.Numerics;

var adjacency=new Dictionary<string,string[]>{["A"]=new[]{"A","A"},["Remote"]=Array.Empty<string>()};
// Your code starts here.
/* TODO: complete the task. */
// Your code ends here.

In [1]:
using System;
using System.Linq;
using System.Collections.Generic;
using System.Numerics;

// Solution
var adjacency=new Dictionary<string,string[]>{["A"]=new[]{"A","A"},["Remote"]=Array.Empty<string>()};
var graph=new GraphWalk(adjacency);adjacency["A"]=new[]{"Remote"};
var bfs=graph.Bfs("A");
Console.WriteLine($"order={string.Join(",",bfs.Order)}, arcs={bfs.EdgesExamined}, remote path={bfs.PathTo("Remote").Length}");
Console.WriteLine($"components={graph.UndirectedComponents().Length}, source path={string.Join(",",bfs.PathTo("A"))}");
try{new GraphWalk(new Dictionary<string,string[]>{["A"]=new[]{"Unknown"}});}catch(ArgumentException){Console.WriteLine("unknown endpoint rejected");}
public sealed class GraphWalk
{
    private readonly Dictionary<string,string[]> graph;
    private readonly string[] vertices;
    public GraphWalk(Dictionary<string,string[]> adjacency)
    {
        ArgumentNullException.ThrowIfNull(adjacency);
        graph=new Dictionary<string,string[]>(StringComparer.Ordinal);
        foreach(var pair in adjacency)
        {
            if(pair.Value is null)throw new ArgumentException("Null adjacency list.");
            graph.Add(pair.Key,pair.Value.Distinct(StringComparer.Ordinal).ToArray());
        }
        foreach(var neighbors in graph.Values)
            if(neighbors.Any(v=>v is null||!graph.ContainsKey(v)))throw new ArgumentException("Unknown endpoint.");
        vertices=graph.Keys.OrderBy(v=>v,StringComparer.Ordinal).ToArray();
    }
    public sealed class Layers
    {
        public string Source { get; }
        public string[] Order { get; }
        public IReadOnlyDictionary<string,int?> Distance { get; }
        public IReadOnlyDictionary<string,string?> Previous { get; }
        public long EdgesExamined { get; }
        internal Layers(string source,string[] order,Dictionary<string,int?> distance,Dictionary<string,string?> previous,long edges)
        {
            Source=source;Order=order;EdgesExamined=edges;
            Distance=new System.Collections.ObjectModel.ReadOnlyDictionary<string,int?>(distance);
            Previous=new System.Collections.ObjectModel.ReadOnlyDictionary<string,string?>(previous);
        }
        public string[] PathTo(string target)
        {
            if(!Distance.ContainsKey(target))throw new KeyNotFoundException(target);
            if(Distance[target] is null)return Array.Empty<string>();
            var path=new Stack<string>();string? at=target;
            while(at is not null)
            {
                if(path.Count>=Distance.Count)throw new InvalidOperationException("Predecessor cycle.");
                path.Push(at);at=Previous[at];
            }
            if(path.Peek()!=Source)throw new InvalidOperationException("Invalid source path.");
            return path.ToArray();
        }
    }
    public Layers Bfs(string source,Action<string>? trace=null)
    {
        if(!graph.ContainsKey(source))throw new KeyNotFoundException(source);
        var distance=vertices.ToDictionary(v=>v,_=>(int?)null,StringComparer.Ordinal);
        var previous=vertices.ToDictionary(v=>v,_=>(string?)null,StringComparer.Ordinal);
        var queue=new Queue<string>();var order=new List<string>();long edges=0;
        distance[source]=0;queue.Enqueue(source);
        while(queue.Count>0)
        {
            string at=queue.Dequeue();order.Add(at);trace?.Invoke($"visit {at}: layer={distance[at]}");
            foreach(string next in graph[at])
            {
                edges++;
                if(distance[next] is not null)continue;
                distance[next]=distance[at]!.Value+1;previous[next]=at;queue.Enqueue(next);
            }
        }
        return new Layers(source,order.ToArray(),distance,previous,edges);
    }
    public record Depth(string[] Preorder,string[] FinishOrder,long EdgesExamined);
    public Depth Dfs(string source,Action<string>? trace=null)
    {
        if(!graph.ContainsKey(source))throw new KeyNotFoundException(source);
        var color=vertices.ToDictionary(v=>v,_=>0,StringComparer.Ordinal);
        var stack=new Stack<(string Vertex,int Next)>();var pre=new List<string>();var finish=new List<string>();long edges=0;
        Enter(source);
        while(stack.Count>0)
        {
            var frame=stack.Pop();
            if(frame.Next==graph[frame.Vertex].Length)
            {color[frame.Vertex]=2;finish.Add(frame.Vertex);trace?.Invoke($"exit {frame.Vertex}");continue;}
            stack.Push((frame.Vertex,frame.Next+1));string next=graph[frame.Vertex][frame.Next];edges++;
            if(color[next]==0)Enter(next);
            else if(color[next]==1)trace?.Invoke($"active edge {frame.Vertex}->{next}");
        }
        return new Depth(pre.ToArray(),finish.ToArray(),edges);
        void Enter(string vertex){color[vertex]=1;pre.Add(vertex);stack.Push((vertex,0));trace?.Invoke($"enter {vertex}");}
    }
    public string[][] UndirectedComponents()
    {
        var neighbors=graph.ToDictionary(p=>p.Key,p=>p.Value.ToHashSet(StringComparer.Ordinal),StringComparer.Ordinal);
        foreach(var pair in graph)
            foreach(string next in pair.Value)
                if(!neighbors[next].Contains(pair.Key))throw new ArgumentException("Components require symmetric adjacency.");
        var seen=new HashSet<string>(StringComparer.Ordinal);var groups=new List<string[]>();
        foreach(string root in vertices)
        {
            if(!seen.Add(root))continue;
            var queue=new Queue<string>();var group=new List<string>();queue.Enqueue(root);
            while(queue.Count>0)
            {
                string at=queue.Dequeue();group.Add(at);
                foreach(string next in graph[at])if(seen.Add(next))queue.Enqueue(next);
            }
            groups.Add(group.ToArray());
        }
        return groups.ToArray();
    }
}


order=A, arcs=1, remote path=0
components=2, source path=A
unknown endpoint rejected


```{rubric} Footnotes
```
[^1]: BFS minimizes edge count. Equal positive edge costs preserve the same ordering of path costs; arbitrary weights require a different algorithm.
[^2]: A directed graph needs a deliberately chosen notion such as weak or strong connectivity; neither is computed by this undirected component method.